# 用 Agent 编排多步检索（Agentic RAG）

本页实现一个受约束、可审计的 Agentic RAG：模型先规划检索目标和工具，执行 dense、BM25 或 hybrid 检索，观察候选证据，再判断是否需要补查，最后核对答案。它对应 `Plan → Act → Observe → Reflect/Repair → Answer` 闭环，而不是只给普通检索流程换一个 Agent 名字。

这里把 Agentic RAG 限定为单次请求内的检索智能体（Agentic Retrieval）：工具集合、最大动作数和补查预算都固定，所有规划、审查和补查响应使用显式 JSON 契约。契约不满足就立即抛出清晰异常，不重试、不切换模型，也不使用 fallback。当前示例只使用《南瓜书》这一资料源；真正的跨来源路由与会话状态由本章的多轮多来源助手单独演示。

## 1. 统一调用边界：先固定可观察的契约

先加载本章数据和检索工具，再定义唯一的模型调用入口 `call_glm_once`。本入口每次只发出一次 `glm-4-flash` 请求，`max_retries=0`，网络、解析或契约错误直接抛出；因此后面的每个阶段都能在 trace 中对应到一次明确调用。阶段之间的独立性只指不同的请求，本页没有跨模型语义 reviewer。


In [1]:
import json
import re
import sys
import unicodedata
from pathlib import Path

def find_tutorial_root(start: Path) -> Path:
    for folder in [start, *start.parents, start / 'notebook' / 'C7 高级 RAG 技巧']:
        if (folder / 'data' / 'dataset/manifest.json').is_file() and (folder / 'common' / 'eval_utils.py').is_file():
            return folder
    raise FileNotFoundError('找不到教程数据目录，请从教程所在目录运行')

TUTORIAL_ROOT = find_tutorial_root(Path.cwd().resolve())
sys.path.insert(0, str(TUTORIAL_ROOT))

from common.eval_utils import emit_tutorial_audit, normalize_text
from common.nontraining_utils import (
    answer_prompt,
    build_bm25_page_search,
    build_reused_chunk_search,
    evidence_payload,
    format_context,
    load_annotation,
    load_pdf_pages,
    load_query_only,
    load_zhipuai_api_key,
    rank_and_coverage,
)
from common.dataset import load_search_evidence

AGENTIC_TOOLS = frozenset({'dense', 'bm25', 'hybrid'})


## 2. Plan：把问题拆成可验证的动作

规划器只能返回 `goal + actions`。每个 action 固定为 `action_N / tool / query / purpose`，并转成同一个稳定的 requirement。解析器在进入检索前拒绝空动作、未知工具、重复 query、跳号 ID 或多余字段；这一步失败就停止，不会用默认计划或悄悄重试。


In [2]:

def call_glm_once(prompt: str, *, max_tokens: int = 900) -> str:
    '''Make exactly one glm-4-flash request; all failures propagate.'''
    from zhipuai import ZhipuAI

    client = ZhipuAI(api_key=load_zhipuai_api_key(), max_retries=0)
    response = client.chat.completions.create(
        model='glm-4-flash',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=max_tokens,
        timeout=60,
    )
    choices = getattr(response, 'choices', None)
    if not choices:
        raise RuntimeError('glm-4-flash 没有返回 choices')
    content = getattr(getattr(choices[0], 'message', None), 'content', None)
    if not isinstance(content, str) or not content.strip():
        raise RuntimeError('glm-4-flash 返回空文字')
    return content.strip()

def _strict_json_object(raw: object, stage: str) -> dict:
    if not isinstance(raw, str) or not raw.strip():
        raise ValueError(f'{stage} 必须返回非空 JSON 对象')
    text = raw.strip()
    if text.startswith('```'):
        lines = text.splitlines()
        if len(lines) < 3 or lines[0].strip().lower() not in {'```', '```json'} or lines[-1].strip() != '```':
            raise ValueError(f'{stage} 的 JSON 围栏不完整')
        text = chr(10).join(lines[1:-1]).strip()
    if '```' in text:
        raise ValueError(f'{stage} 包含 JSON 之外的围栏或文字')

    def reject_duplicate_keys(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f'{stage} JSON 含重复字段：{key}')
            result[key] = value
        return result

    try:
        value = json.loads(text, object_pairs_hook=reject_duplicate_keys)
    except json.JSONDecodeError as error:
        raise ValueError(f'{stage} 不是合法 JSON：{error}') from error
    except TypeError as error:
        raise ValueError(f'{stage} 不是可解析 JSON：{error}') from error
    if not isinstance(value, dict):
        raise ValueError(f'{stage} 必须是 JSON 对象')
    return value

def _strict_nonempty_string(value: object, field: str, stage: str) -> str:
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f'{stage}.{field} 必须是非空字符串')
    return value.strip()

def _strict_string_list(value: object, field: str, stage: str, *, min_items: int, max_items: int, unique: bool) -> list[str]:
    if not isinstance(value, list) or not min_items <= len(value) <= max_items:
        raise ValueError(f'{stage}.{field} 必须是长度 {min_items}..{max_items} 的字符串列表')
    result = []
    for index, item in enumerate(value):
        if not isinstance(item, str) or not item.strip():
            raise ValueError(f'{stage}.{field}[{index}] 必须是非空字符串')
        result.append(item.strip())
    if unique and len(result) != len(set(result)):
        raise ValueError(f'{stage}.{field} 不得包含重复字符串')
    return result

_ACTION_ID_PATTERN = re.compile(r'action_[1-9][0-9]*\Z')

def parse_agentic_plan(raw: str) -> dict:
    stage = 'Agentic Plan'
    payload = _strict_json_object(raw, stage)
    if set(payload) != {'goal', 'actions'}:
        raise ValueError(f'{stage} 字段必须精确为 goal、actions')
    goal = _strict_nonempty_string(payload['goal'], 'goal', stage)
    actions = payload['actions']
    if not isinstance(actions, list) or not 1 <= len(actions) <= 3:
        raise ValueError(f'{stage}.actions 必须包含 1..3 个动作')
    parsed_actions = []
    for index, action in enumerate(actions):
        action_stage = f'{stage}.actions[{index}]'
        if not isinstance(action, dict) or set(action) != {'action_id', 'tool', 'query', 'purpose'}:
            raise ValueError(f'{action_stage} 字段必须精确为 action_id、tool、query、purpose')
        action_id = _strict_nonempty_string(action['action_id'], 'action_id', action_stage)
        expected_action_id = f'action_{index + 1}'
        if not _ACTION_ID_PATTERN.fullmatch(action_id) or action_id != expected_action_id:
            raise ValueError(f'{action_stage}.action_id 必须按规划顺序稳定编号为 {expected_action_id}')
        tool = _strict_nonempty_string(action['tool'], 'tool', action_stage)
        if tool not in AGENTIC_TOOLS:
            raise ValueError(f'{action_stage}.tool 不是允许的检索工具：{tool}')
        query = _strict_nonempty_string(action['query'], 'query', action_stage)
        purpose = _strict_nonempty_string(action['purpose'], 'purpose', action_stage)
        parsed_actions.append({'action_id': action_id, 'tool': tool, 'query': query, 'purpose': purpose})
    action_queries = [action['query'] for action in parsed_actions]
    if len(action_queries) != len(set(action_queries)):
        raise ValueError(f'{stage}.actions.query 不得重复')
    return {'goal': goal, 'actions': parsed_actions}

def plan_requirements(plan: dict) -> list[dict]:
    """Turn each planned action into one stable, independently verifiable requirement."""
    actions = plan.get('actions') if isinstance(plan, dict) else None
    if not isinstance(actions, list) or not actions:
        raise ValueError('规划必须包含可验证的独立 requirements')
    return [
        {'requirement_id': action['action_id'], 'query': action['query'], 'purpose': action['purpose']}
        for action in actions
    ]


## 3. Verify 的输入契约：观察之后再判断缺口

每个检索 action 之后先保存真实候选，再由 verifier 只在 canonical evidence 上把已规划 requirement 分到 `covered` 或 `missing`，每个 ID 恰好一次。`sufficient` 必须等价于 `missing == []`。这里的 verify 是同一 `glm-4-flash` 的另一次独立请求，不是跨模型审核；它不能使用常识补证据，也不能临时增加需求。repair 解析器只允许一个新的检索动作。


In [3]:

def _requirement_key(value: str) -> str:
    """Normalize verifier labels so paraphrased overlap cannot pass."""
    text = unicodedata.normalize('NFKC', value).casefold()
    text = ''.join(character for character in text if not character.isspace() and not unicodedata.category(character).startswith(('P', 'S')))
    if 'mds' in text and ('目标' in text or '降维' in text):
        return 'mds:target'
    if 'ksvd' in text and any(token in text for token in ('迭代', '更新', '两步', '稀疏', '字典')):
        return 'ksvd:update_steps'
    text = re.sub(r'(?:算法)?(?:的)?(?:降维)?目标', '目标', text)
    return text

def parse_agentic_verify(raw: str, requirement_ids: list[str] | None = None) -> dict:
    stage = 'Agentic Verify'
    payload = _strict_json_object(raw, stage)
    if set(payload) != {'sufficient', 'covered', 'missing'}:
        raise ValueError(f'{stage} 字段必须精确为 sufficient、covered、missing')
    sufficient = payload['sufficient']
    if type(sufficient) is not bool:
        raise ValueError(f'{stage}.sufficient 必须是 JSON boolean')
    covered = _strict_string_list(payload['covered'], 'covered', stage, min_items=0, max_items=12, unique=True)
    missing = _strict_string_list(payload['missing'], 'missing', stage, min_items=0, max_items=12, unique=True)
    covered_keys = [_requirement_key(item) for item in covered]
    missing_keys = [_requirement_key(item) for item in missing]
    if '' in covered_keys or '' in missing_keys:
        raise ValueError(f'{stage} covered/missing 规范化标签不能为空')
    if len(covered_keys) != len(set(covered_keys)):
        raise ValueError(f'{stage}.covered 含语义重复要点')
    if len(missing_keys) != len(set(missing_keys)):
        raise ValueError(f'{stage}.missing 含语义重复要点')
    if set(covered_keys) & set(missing_keys):
        raise ValueError(f'{stage}.covered 与 missing 不能包含同一规范化要点')
    if sufficient != (not missing):
        raise ValueError(f'{stage}.sufficient 必须严格等价于 missing 是否为空')
    if sufficient and not covered:
        raise ValueError(f'{stage} 足够时 covered 必须非空')
    if requirement_ids is not None:
        expected_ids = _strict_string_list(requirement_ids, 'requirement_ids', stage, min_items=1, max_items=12, unique=True)
        if any(not _ACTION_ID_PATTERN.fullmatch(item) for item in expected_ids):
            raise ValueError(f'{stage}.requirement_ids 必须使用稳定 action_N ID')
        observed_ids = covered + missing
        if (len(observed_ids) != len(expected_ids)
                or len(set(observed_ids)) != len(observed_ids)
                or set(observed_ids) != set(expected_ids)):
            raise ValueError(f'{stage} covered/missing 必须把每个规划 requirement/action ID 恰好分配一次')
    return {'sufficient': sufficient, 'covered': covered, 'missing': missing}

def parse_agentic_repair(raw: str, used_queries: list[str]) -> dict:
    stage = 'Agentic Repair'
    payload = _strict_json_object(raw, stage)
    if set(payload) != {'tool', 'query', 'purpose'}:
        raise ValueError(f'{stage} 字段必须精确为 tool、query、purpose')
    tool = _strict_nonempty_string(payload['tool'], 'tool', stage)
    if tool not in AGENTIC_TOOLS:
        raise ValueError(f'{stage}.tool 不是允许的检索工具：{tool}')
    query = _strict_nonempty_string(payload['query'], 'query', stage)
    purpose = _strict_nonempty_string(payload['purpose'], 'purpose', stage)
    query_key = _query_key(query)
    used_query_keys = {_query_key(item) for item in used_queries}
    if query_key in used_query_keys:
        raise ValueError(f'{stage}.query 必须不同于已用 query（忽略空白和标点）')
    return {'tool': tool, 'query': query, 'purpose': purpose}

def _query_key(value: str) -> str:
    text = unicodedata.normalize('NFKC', value).casefold()
    return ''.join(character for character in text if not character.isspace() and not unicodedata.category(character).startswith(('P', 'S')))


## 4. Answer 契约：证据绑定先于回答

观察结果要绑定到本轮真实命中的 canonical quote，不能只凭页码或主题名。最终答案 raw 只能是 `{status, claims}`；parser 检查每条 `statement` 是否由所列 1..4 个 quote 按顺序逐字拼接（只移除空白），再确定性生成 `answer`。任何伪造 evidence ID、语义改写、重复字段或 unsupported claim 都在边界处抛错。


In [4]:

def _compact_text(value: object) -> str:
    return re.sub(r'\s+', '', normalize_text(value))

def _canonical_matches(hit) -> list[dict]:
    return list(candidate_catalog_from_rows([hit], CANONICAL_EVIDENCE).values())

def _hit_key(hit):
    chunk_id = getattr(hit, 'chunk_id', None)
    if chunk_id:
        return ('chunk', str(chunk_id))
    return ('page_text', int(hit.page), _compact_text(hit.text))

def merge_action_evidence(action_hits: list[list], repair_hits: list | None = None, *, limit: int = 8) -> list:
    """Merge plan groups round-robin, then repair, without starving an action."""
    if not isinstance(limit, int) or isinstance(limit, bool) or limit <= 0:
        raise ValueError('证据合并 limit 必须是正整数')
    groups = [list(group) for group in action_hits]
    if repair_hits is not None:
        groups.append(list(repair_hits))
    if not groups or not any(groups):
        raise ValueError('没有可合并的真实检索候选')
    offsets = [0] * len(groups)
    seen = set()
    merged = []
    while len(merged) < limit:
        progress = False
        for group_index, group in enumerate(groups):
            while offsets[group_index] < len(group):
                hit = group[offsets[group_index]]
                offsets[group_index] += 1
                key = _hit_key(hit)
                if key in seen:
                    continue
                seen.add(key)
                merged.append(hit)
                progress = True
                break
            if len(merged) >= limit:
                break
        if not progress:
            break
    if not merged:
        raise ValueError('真实检索候选在去重后为空')
    return merged

def observed_payload(hits: list) -> list[dict]:
    payload = []
    for item in evidence_payload(hits):
        hit = hits[len(payload)]
        item['matched_evidence_ids'] = [row['evidence_id'] for row in _canonical_matches(hit)]
        payload.append(item)
    return payload

def candidate_catalog_from_rows(hits: list, evidence_rows: list[dict]) -> dict[str, dict]:
    """Bind every retrieved hit to all matching canonical quotes, without topic filters."""
    catalog = {}
    for hit in hits:
        hit_text = _compact_text(hit.text)
        for row in evidence_rows:
            # fixed_token_chunk 只是检索载荷，不能冒充可引用事实。
            if str(row.get('evidence_id', '')).startswith('evi_ft_chunk_'):
                continue
            if int(row['page']) != int(hit.page):
                continue
            quote = str(row['quote'])
            if _compact_text(quote) not in hit_text:
                continue
            evidence_id = str(row['evidence_id'])
            if evidence_id not in catalog:
                catalog[evidence_id] = {
                    'evidence_id': evidence_id,
                    'page': int(row['page']),
                    'quote': quote,
                }
    return catalog

def candidate_evidence_catalog(hits: list) -> dict[str, dict]:
    return candidate_catalog_from_rows(hits, CANONICAL_EVIDENCE)

def _quote_surface(value: object) -> str:
    """Fail-closed：NFKC 后只移除空白，保留全部语义和版式字符。"""
    text = unicodedata.normalize('NFKC', str(value or ''))
    return ''.join(character for character in text if not character.isspace())

def _claim_directly_supported(statement: str, evidence_rows: list[dict]) -> bool:
    if not isinstance(statement, str) or not statement.strip() or not evidence_rows:
        return False
    # The claim must be a faithful concatenation of the cited canonical quotes.
    # Removing only whitespace keeps every punctuation/operator change fail-closed.
    expected = ''.join(_quote_surface(row['quote']) for row in evidence_rows)
    actual = _quote_surface(statement)
    return bool(expected) and actual == expected

def parse_agentic_answer(raw: str, candidate_catalog: dict[str, dict], *, expected_sufficient: bool) -> dict:
    stage = 'Agentic Answer'
    payload = _strict_json_object(raw, stage)
    if set(payload) != {'status', 'claims'}:
        raise ValueError(f'{stage} 字段必须精确为 status、claims')
    status = payload['status']
    if status not in {'answered', 'insufficient'}:
        raise ValueError(f'{stage}.status 只能是 answered 或 insufficient')
    if type(expected_sufficient) is not bool:
        raise TypeError('expected_sufficient 必须是 boolean')
    if expected_sufficient and status != 'answered':
        raise ValueError(f'{stage} 资料已足够时必须输出 status=answered')
    if not expected_sufficient and status != 'insufficient':
        raise ValueError(f'{stage} 资料不足时必须输出 status=insufficient')
    claims = payload['claims']
    if not isinstance(claims, list):
        raise ValueError(f'{stage}.claims 必须是列表')
    if status == 'insufficient':
        if claims:
            raise ValueError(f'{stage} status=insufficient 时 claims 必须为空')
        return {'answer': '资料不足，无法可靠回答。', 'status': status, 'claims': []}
    if not claims:
        raise ValueError(f'{stage} status=answered 时至少需要一条 claim')
    hydrated = []
    for index, claim in enumerate(claims):
        claim_stage = f'{stage}.claims[{index}]'
        if not isinstance(claim, dict) or set(claim) != {'statement', 'evidence_ids'}:
            raise ValueError(f'{claim_stage} 字段必须精确为 statement、evidence_ids')
        statement = _strict_nonempty_string(claim['statement'], 'statement', claim_stage)
        evidence_ids = _strict_string_list(claim['evidence_ids'], 'evidence_ids', claim_stage, min_items=1, max_items=4, unique=True)
        evidence_rows = []
        for evidence_id in evidence_ids:
            evidence = candidate_catalog.get(evidence_id)
            if evidence is None:
                raise ValueError(f'{claim_stage} 引用了本轮真实候选之外的 evidence_id：{evidence_id}')
            evidence_rows.append({'evidence_id': evidence_id, 'page': evidence['page'], 'quote': evidence['quote']})
        if not _claim_directly_supported(statement, evidence_rows):
            raise ValueError(f'{claim_stage} 的完整 statement 没有被所列 canonical quote 直接支持')
        hydrated.append({'statement': statement, 'evidence_ids': evidence_ids, 'evidence': evidence_rows})
    joined_answer = ''.join(item['statement'] for item in hydrated)
    return {'answer': joined_answer, 'status': status, 'claims': hydrated}

def _must_reject(function, *args):
    try:
        function(*args)
    except (TypeError, ValueError):
        return
    raise AssertionError(f'{function.__name__} 错误输入未被拒绝')

_must_reject(parse_agentic_plan, json.dumps({'goal': 'g', 'actions': []}))
_must_reject(parse_agentic_plan, json.dumps({'goal': 'g', 'actions': [{'tool': 'hybrid', 'query': '', 'purpose': 'p'}]}))
_must_reject(parse_agentic_verify, json.dumps({'sufficient': 'false', 'covered': [], 'missing': ['m']}))
_must_reject(parse_agentic_verify, json.dumps({'sufficient': False, 'covered': [], 'missing': []}))
_must_reject(parse_agentic_repair, '{}', [])
_must_reject(parse_agentic_repair, json.dumps({'tool': 'other', 'query': 'q', 'purpose': 'p'}), [])
_must_reject(parse_agentic_repair, json.dumps({'tool': 'dense', 'query': 'q', 'purpose': 'p'}), ['q'])
_must_reject(parse_agentic_verify, json.dumps({'sufficient': False, 'covered': ['MDS 算法的目标'], 'missing': ['MDS 的降维目标']}))


## 5. Act → Observe：每个动作都留下可审计的检索记录

`run_tool` 是受限工具执行层：只允许 dense、BM25、hybrid；hybrid 在 hit/chunk 粒度做 RRF，不能把同页不同片段误当成同一个结果。每个 action 的工具、query、页码、候选文本和可绑定 evidence ID 都进入 trace；合并时按 action round-robin，避免补查或某一个 action 挤掉其他需求。


In [5]:

CASE_IDS = ['agentic_mds_ksvd', 'agentic_kpca_centering']
queries = load_query_only(CASE_IDS)
CANONICAL_EVIDENCE = load_search_evidence()
pages = load_pdf_pages()
dense = build_reused_chunk_search()
bm25 = build_bm25_page_search(pages)

def rank_fusion(dense_hits, bm25_hits, top_k: int = 4, rrf_k: int = 60):
    """Fuse at the shared hit/chunk grain; page labels are not identities."""
    if not isinstance(top_k, int) or isinstance(top_k, bool) or top_k <= 0:
        raise ValueError('Hybrid RRF top_k 必须是正整数')
    if not isinstance(rrf_k, int) or isinstance(rrf_k, bool) or rrf_k < 0:
        raise ValueError('Hybrid RRF rrf_k 必须是非负整数')
    scores = {}
    by_hit = {}
    for ranked_hits in (dense_hits, bm25_hits):
        for rank, hit in enumerate(ranked_hits, 1):
            key = _hit_key(hit)
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
            if key not in by_hit:
                by_hit[key] = hit
    ordered_keys = sorted(by_hit, key=lambda key: (-scores[key], key))[:top_k]
    return [by_hit[key] for key in ordered_keys]

def run_tool(tool_name: str, query: str, top_k: int = 4):
    if tool_name == 'hybrid':
        dense_hits = dense(query, top_k=max(8, top_k))
        bm25_hits = bm25(query, top_k=max(8, top_k))
        fused_hits = rank_fusion(dense_hits, bm25_hits, top_k=top_k)
        return fused_hits, {'dense_candidates': observed_payload(dense_hits), 'bm25_candidates': observed_payload(bm25_hits), 'fused': observed_payload(fused_hits)}
    if tool_name not in AGENTIC_TOOLS:
        raise ValueError(f'未知检索工具：{tool_name}')
    hits = (dense if tool_name == 'dense' else bm25)(query, top_k=top_k)
    return hits, {'candidates': observed_payload(hits)}

def baseline_pipeline(question: str):
    hits = dense(question, top_k=4)
    answer = call_glm_once(answer_prompt(question, format_context(hits)))
    return hits, answer

def verify_once(question: str, hits: list, phase: str, requirements: list[dict]) -> dict:
    if not isinstance(requirements, list) or not requirements:
        raise ValueError('verify 必须接收非空的 planned requirements')
    requirement_ids = [item.get('requirement_id') for item in requirements if isinstance(item, dict)]
    if len(requirement_ids) != len(requirements) or any(not isinstance(item, str) for item in requirement_ids):
        raise ValueError('planned requirements 必须包含稳定 requirement_id')
    verify_schema = {'sufficient': True, 'covered': requirement_ids, 'missing': []}
    canonical_catalog = candidate_evidence_catalog(hits)
    verify_raw = call_glm_once(
        '你是证据审查器，只能依据用户问题和候选资料判断资料是否足够。不要凭常识补充资料。'
        + '本轮真实命中的可核验 canonical evidence（字段只有 evidence_id、page、quote；先依据这些逐字证据判断覆盖情况，未列出的证据不能算已覆盖）：'
        + json.dumps(list(canonical_catalog.values()), ensure_ascii=False) + chr(10)
        + '计划中的独立 requirements（每个 requirement_id 对应一个 action，ID 稳定且不可改写）：'
        + json.dumps(requirements, ensure_ascii=False) + chr(10)
        + '只判断这些已规划 requirement_id 是否被本轮 canonical evidence 直接覆盖；不要临时增加要点，也不要把一个 requirement 拆成多个标签。每个 requirement_id 必须恰好进入 covered 或 missing 其中一列一次。'
        + '这是机器可读接口：禁止 Markdown、解释文字、reason、explanation、confidence、goal、phase，以及任何未列出的字段。sufficient 必须是 JSON boolean；covered 和 missing 必须是长度 0..12 的 requirement_id 字符串列表；sufficient 必须严格等价于 missing 为空。最终对象的键集合必须恰好是 {sufficient, covered, missing}，只输出这个合法 JSON 对象：'
        + json.dumps(verify_schema, ensure_ascii=False) + chr(10)
        + '用户问题：' + question + chr(10)
        + '审查阶段：' + phase + chr(10)
        + '候选资料（text 是真实检索片段，matched_evidence_ids 仅列出可由该片段逐字绑定的 canonical evidence；仅作为上述 canonical evidence 的补充上下文）：'
        + json.dumps(observed_payload(hits), ensure_ascii=False),
        max_tokens=260,
    )
    parsed = parse_agentic_verify(verify_raw, requirement_ids)
    return {'step': 'verify', 'phase': phase, 'raw': verify_raw, 'parsed': parsed, 'pages': [hit.page for hit in hits]}



## 6. 控制循环：Verify → 必要时 Repair → 再 Verify

`agentic_pipeline` 的顺序是：一次 plan → 逐 action 执行并观察 → 首轮 verify → 仅在 `sufficient=false` 时消耗一次 repair 预算 → 执行新 query → `after_repair` 再 verify → 最后才生成答案。repair 的 query 必须不同于已用 query；再验证仍不足时只能输出固定拒答，不能 fallback 或跳过验证。


In [6]:
def agentic_pipeline(question: str):
    plan_schema = {'goal': '要找全的证据目标', 'actions': [{'action_id': 'action_1', 'tool': 'dense|bm25|hybrid', 'query': '非空检索短语', 'purpose': '一个独立回答需求'}]}
    plan_raw = call_glm_once(
        '你是 RAG 检索规划器。把用户问题拆成 1 到 3 个互不重复、可独立核验的回答需求；每个 action 只服务一个需求。action_id 必须严格按顺序写成 action_1、action_2、action_3，不能改名、重复或跳号。tool 只能是 dense、bm25、hybrid；query 和 purpose 都必须是非空字符串。只输出一个合法 JSON 对象，字段必须精确匹配这个 schema：' + json.dumps(plan_schema, ensure_ascii=False) + chr(10) + '用户问题：' + question,
        max_tokens=420,
    )
    plan = parse_agentic_plan(plan_raw)
    requirements = plan_requirements(plan)
    trace = [{'step': 'plan', 'raw': plan_raw, 'parsed': plan, 'requirements': requirements}]
    action_groups = []
    used_queries = []
    for action in plan['actions']:
        tool_name = action['tool']
        query = action['query']
        action_top_k = 4
        hits, retrieval_details = run_tool(tool_name, query, top_k=action_top_k)
        action_groups.append(hits)
        used_queries.append(query)
        trace.append({'step': 'retrieve', 'action_id': action['action_id'], 'requirement_id': action['action_id'], 'tool': tool_name, 'query': query, 'purpose': action['purpose'], 'pages': [hit.page for hit in hits], 'evidence': observed_payload(hits), 'candidates': retrieval_details})
    merged = merge_action_evidence(action_groups, limit=8)
    initial_verify_stage = verify_once(question, merged, 'initial', requirements)
    trace.append(initial_verify_stage)
    final_verify_stage = initial_verify_stage
    repair_stage = None
    verify_after_repair_stage = None
    if not initial_verify_stage['parsed']['sufficient']:
        repair_schema = {'tool': 'dense|bm25|hybrid', 'query': '不同于已用 query 的非空检索短语', 'purpose': '非空补查目的'}
        repair_raw = call_glm_once(
            '你是 RAG 检索修复器。证据不足时只生成一次不同于已用 query 的补查动作。tool 只能是 dense、bm25、hybrid；query 和 purpose 必须是非空字符串；不得输出空对象或其它字段。只输出字段精确匹配这个 schema 的合法 JSON：' + json.dumps(repair_schema, ensure_ascii=False) + chr(10) + '用户问题：' + question + chr(10) + '审查结果：' + json.dumps(initial_verify_stage['parsed'], ensure_ascii=False) + chr(10) + '已用 query：' + json.dumps(used_queries, ensure_ascii=False),
            max_tokens=300,
        )
        repair = parse_agentic_repair(repair_raw, used_queries)
        repair_hits, repair_details = run_tool(repair['tool'], repair['query'], top_k=4)
        merged = merge_action_evidence(action_groups, repair_hits, limit=8)
        repair_stage = {'step': 'repair', 'raw': repair_raw, 'parsed': repair, 'pages': [hit.page for hit in repair_hits], 'evidence': observed_payload(repair_hits), 'candidates': repair_details}
        trace.append(repair_stage)
        verify_after_repair_stage = verify_once(question, merged, 'after_repair', requirements)
        trace.append(verify_after_repair_stage)
        final_verify_stage = verify_after_repair_stage
    final_verify = final_verify_stage['parsed']
    stop_reason = 'verified_sufficient' if final_verify['sufficient'] else 'insufficient_after_repair'
    candidate_catalog = candidate_evidence_catalog(merged)
    answer_schema = {'status': 'answered|insufficient', 'claims': [{'statement': '直接复制一个 canonical quote 原文（只可去空白，不得改写）', 'evidence_ids': ['1..4 个候选 evidence_id'] }, {'statement': '另一个独立 canonical quote 原文', 'evidence_ids': ['1..4 个候选 evidence_id'] }]}
    answer_contract_instruction = (
        '硬约束：最终验证 sufficient=false。你不能回答用户问题，也不能写任何事实陈述；必须原样输出一个 JSON 对象 {"status":"insufficient","claims":[]}，不得改变 status，不得添加 claim，也不得添加 answer 字段。'
        if not final_verify['sufficient']
        else '硬约束：最终验证 sufficient=true。这是逐字字符串契约，不是自然语言问答。只能输出 status=answered 和 claims；不得输出 answer 字段。claims 至少包含一条，可按独立事实或步骤拆成多条。每条 claim 的 statement 必须由其 evidence_ids 对应的 1..4 个 canonical quote 直接支持；evidence_ids 必须是候选 catalog 中实际出现的唯一 evidence_id，只列直接支撑本条 statement 的证据，不要求单条 claim 或全部 claims 引用全部候选。statement 必须直接复制所列 quote 按 evidence_ids 顺序的原始字符串，只允许移除版式空白；冒号、逗号、句号、运算符、括号、下划线、连字符和大小写都不得改动，不得添加、删减、反转或补充 quote 没有的事实；无法由 quote 直接支持的内容必须省略。特别注意：quote 中的冒号和逗号也是原文字符，例如 K-SVD quote 的"分两步:"中的冒号、Codebook quote 的"Stage, 在"中的逗号都必须原样出现在对应 statement，漏掉即错误。输出 JSON 前先检查每条 statement 是否为所列 quote 字符串按顺序拼接后仅移除空白的结果；answer 不在模型输出中，由 parser 在 hydrate 后确定性生成。不要把用户问题中的词当成证据。若资料支持不足，严格输出 status=insufficient、claims=[]。'
    )
    copy_checklist = chr(10).join(
        'ID=' + str(item['evidence_id']) + '；statement 必须逐字复制 quote=' + json.dumps(str(item['quote']), ensure_ascii=False)
        for item in candidate_catalog.values()
    ) if final_verify['sufficient'] else ''
    answer_raw = call_glm_once(
        '你是最终答案生成器。这是机器可读接口：严格输出一个合法 JSON 对象，字段必须精确匹配这个 schema：' + json.dumps(answer_schema, ensure_ascii=False) + chr(10)
        + '用户问题：' + question + chr(10)
        + '最终证据审查：' + json.dumps(final_verify, ensure_ascii=False) + chr(10)
        + '候选资料上下文（以下是不可变 canonical quote 字符串；不得把其它检索片段中的补充解释写入 statement；每条 claim 可选择其中 1..4 个直接支撑自身 statement 的 evidence_id）：' + copy_checklist + chr(10)
        + '每条 claim 只能从上述 canonical evidence 选择 1..4 个 evidence_id；可以按独立事实拆成多条 claim，不要求某一条 claim 或全部 claims 引用整个候选列表。每条 claim 的 evidence_ids 只列直接支撑该 statement 所需的 1..4 个唯一 evidence_id；若合并多个 ID，statement 必须完整复述所列 quote 按顺序拼接的全部内容。statement 不是自然语言改写，而是把所列 quote 的字符串值逐字符复制；只允许移除版式空白，冒号、逗号、句号、运算符、括号、下划线、连字符和大小写都必须逐字符保留。不能把其它 quote 的 ID 挂在只复述部分 quote 的 statement 上，剩余 quote 应另建 claim。特别注意：quote 中的冒号和逗号不能省略；evi_98b18037d257 的 quote 必须保留"分两步:"中的冒号，evi_e0f0422335a4 的 quote 必须保留"Stage, 在"中的逗号，不能按自然语言习惯删除。模型只输出 status 和 claims，不输出 answer；parser 会在 hydrate 后把 claims.statement 按顺序无分隔符拼接成 answer。'
        + '禁止 Markdown、解释文字、reason、answer、quote 字段，以及任何未列出的字段。最终对象的键集合必须恰好是 {status, claims}。' + answer_contract_instruction
        + (chr(10) + '最后再按下面不可变清单逐字复制 quote 到对应 statement；不要重新措辞或删除任何冒号/逗号。这是复制任务而非问答：只输出 status+claims，不输出 answer；parser 会把 claims.statement 无分隔符拼接为 answer：' + copy_checklist + chr(10) + '严格执行覆盖前述所有说明：可以保留或删除空格，但不能删除、增加或替换任何非空白字符；只完成 claims，不要总结、改写或添加句号。' if copy_checklist else ''),
        max_tokens=1200,
    )
    answer = parse_agentic_answer(answer_raw, candidate_catalog, expected_sufficient=final_verify['sufficient'])
    trace.extend([
        {'step': 'stop', 'reason': stop_reason, 'pages': [hit.page for hit in merged], 'final_verify': final_verify},
        {'step': 'answer', 'raw': answer_raw, 'parsed': answer, 'pages': [hit.page for hit in merged]},
    ])
    stage_outputs = {
        'plan': {'raw': plan_raw, 'parsed': plan},
        'verify_initial': initial_verify_stage,
        'verify_after_repair': verify_after_repair_stage,
        'verify': final_verify_stage,
        'candidate_catalog': candidate_catalog,
        'repair': repair_stage,
        'answer': {'raw': answer_raw, 'parsed': answer},
        'trace': trace,
        'stage_call_counts': {'plan': 1, 'verify_initial': 1, 'repair': int(repair_stage is not None), 'verify_after_repair': int(verify_after_repair_stage is not None), 'answer': 1},
    }
    return merged, answer, stage_outputs



## 7. 运行两个真实案例并保存审计

最后一个单元才读取固定回归题、运行 baseline 与 Agentic 流程，并把每阶段 raw、parsed、trace、候选 catalog 和调用计数写入审计 MIME。当前保存的两个真实案例均在首轮 verify 已足够，因此真实 trace 没有 repair；repair→reverify 的控制流由本章 `tests/c7/test_agentic_rag_contracts.py` 的 mock/unit 合约测试覆盖，不能把测试结果写成真实模型曾触发。


In [7]:
records = []
for item in queries:
    before, baseline_answer = baseline_pipeline(item['query'])
    final_hits, _, outputs = agentic_pipeline(item['query'])
    outputs['baseline_answer'] = {'raw': baseline_answer, 'parsed': baseline_answer}
    records.append({'case_id': item['id'], 'query': item['query'], 'before': before, 'after': final_hits, 'model_outputs': outputs})

# 生成和检索完成后才读取评估标注；审计记录保留每一阶段的 raw、parsed 和 trace。
for record in records:
    record['annotation'] = load_annotation(record['case_id'])
    outputs = record['model_outputs']
    before = rank_and_coverage(record['before'], record['annotation']['expected_pages'])
    after = rank_and_coverage(record['after'], record['annotation']['expected_pages'])
    initial_verify = outputs['verify_initial']['parsed']
    verify = outputs['verify']['parsed']
    retrieval_choices = []
    for step in outputs['trace']:
        if step['step'] not in {'retrieve', 'repair'}:
            continue
        parsed = step.get('parsed', {})
        retrieval_choices.append({'tool': parsed.get('tool', step.get('tool')), 'query': parsed.get('query', step.get('query')), 'pages': step.get('pages', [])})
    repair = next((step for step in outputs['trace'] if step['step'] == 'repair'), None)
    missing = verify['missing']
    if repair:
        repair_reason = ('首轮证据不足，缺少：' + '、'.join(initial_verify['missing']) + '；'
                         + ('补查后再次严格验证：证据已足够' if verify['sufficient']
                            else '补查后再次严格验证仍不足，最终拒答'))
    else:
        repair_reason = '首轮证据已足够，无需补查'
    summary = {
        'case_id': record['case_id'],
        'method': '让模型选择检索方式（Agentic Retrieval）',
        'role': 'main' if record['case_id'] == CASE_IDS[0] else 'check',
        'query': record['query'],
        'before': {'pages': before['pages'], 'first_required_rank': before['first_required_rank'], 'required_page_coverage': before['required_page_coverage']},
        'after': {'pages': after['pages'], 'first_required_rank': after['first_required_rank'], 'required_page_coverage': after['required_page_coverage']},
        'retrieval_choices': retrieval_choices,
        'repair_reason': repair_reason,
        'repair': {'tool': repair['parsed']['tool'], 'query': repair['parsed']['query'], 'pages': repair['pages']} if repair else None,
        'model_outputs': outputs,
        'execution_trace': outputs['trace'],
        'stage_call_counts': {'baseline': 1, 'agentic': sum(outputs['stage_call_counts'].values())},
    }
    answer = outputs['answer']['parsed']
    summary['answer'] = answer
    summary['baseline_model_call_count'] = 1
    summary['agentic_model_call_count'] = sum(outputs['stage_call_counts'].values())
    summary['model_call_count'] = summary['baseline_model_call_count'] + summary['agentic_model_call_count']
    print('问题：', record['query'])
    print('改动前：必要页覆盖', before['required_page_coverage'], '，页', before['pages'])
    print('流程摘要（根据实际执行记录）：', summary['repair_reason'])
    print('实际执行的检索问题：首轮（基线 dense）：', record['query'], '；', '；'.join(choice['tool'] + '：' + choice['query'] for choice in retrieval_choices))
    print('模型原始回答（基线，截取）：', outputs['baseline_answer']['raw'][:180])
    print('改动后：必要页覆盖', after['required_page_coverage'], '，页', after['pages'])
    print('模型原始回答（Agentic 最终 JSON，截取）：', outputs['answer']['raw'][:260])
    print('资料量：', len(record['before']), '→', len(record['after']), '个片段')
    print('模型调用次数：基线', summary['baseline_model_call_count'], '次；Agentic Retrieval', summary['agentic_model_call_count'], '次；合计', summary['model_call_count'], '次。')
    print('结论：模型先规划并检查资料；若补查，则再次用 glm-4-flash 严格复核，最终答案遵守 evidence_id/quote 契约。')
    print(chr(10) + '--- 代理式检索对照：' + record['query'] + '（模型=glm-4-flash）---')
    emit_tutorial_audit(summary)


问题： MDS 的降维目标是什么，K-SVD 每轮更新又分为哪两步？
改动前：必要页覆盖 0.5 ，页 [132, 133, 125, 145]
流程摘要（根据实际执行记录）： 首轮证据已足够，无需补查
实际执行的检索问题：首轮（基线 dense）： MDS 的降维目标是什么，K-SVD 每轮更新又分为哪两步？ ； dense：MDS的降维目标是什么；bm25：K-SVD每轮更新分为哪两步
模型原始回答（基线，截取）： MDS（多维尺度分析）的降维目标是保持原始空间中样本之间的距离在低维空间中得以保持。也就是说，降维的目标是“任意两个样本在d′维空间中的欧氏距离等于原始空间中的距离”。

K-SVD（K-means Singular Value Decomposition）算法每轮更新分为以下两步：

1. **K-means步骤**：在这一步中，将数据集中的每个样本分配到
改动后：必要页覆盖 1.0 ，页 [133, 146, 125, 147, 133, 176, 132, 189]
模型原始回答（Agentic 最终 JSON，截取）： ```json
{
  "status": "answered",
  "claims": [
    {
      "statement": "MDS 算法的降维准则是要求原始空间中样本之间的距离在低维空间中得以保持",
      "evidence_ids": ["evi_13bc746d5509"]
    },
    {
      "statement": "K-SVD 算法迭代过程分两步: 第1 步Sparse Coding Stage 就是普通的已知字典矩阵D 的稀疏表示问题",
      "
资料量： 4 → 8 个片段
模型调用次数：基线 1 次；Agentic Retrieval 3 次；合计 4 次。
结论：模型先规划并检查资料；若补查，则再次用 glm-4-flash 严格复核，最终答案遵守 evidence_id/quote 契约。

--- 代理式检索对照：MDS 的降维目标是什么，K-SVD 每轮更新又分为哪两步？（模型=glm-4-flash）---


问题： 核 PCA 的推导为什么要求映射后的样本先中心化？
改动前：必要页覆盖 1.0 ，页 [68, 64, 132, 40]
流程摘要（根据实际执行记录）： 首轮证据已足够，无需补查
实际执行的检索问题：首轮（基线 dense）： 核 PCA 的推导为什么要求映射后的样本先中心化？ ； dense：PCA 推导中样本中心化的原因；bm25：为什么 PCA 需要对样本进行中心化；hybrid：PCA 推导中样本中心化的数学原理
模型原始回答（基线，截取）： 核PCA（核主成分分析）的推导要求映射后的样本先中心化，主要是为了确保计算过程中的一些数学性质能够得到满足，具体原因如下：

1. **均值为零的便利性**：在核PCA中，样本通过核函数映射到高维空间。在高维空间中，直接计算样本的均值可能比较复杂。如果样本中心化，即每个样本的均值被减去，那么在高维空间中的每个样本的“中心”点就是原点（0,0,...,0）。这
改动后：必要页覆盖 1.0 ，页 [132, 129, 118, 123, 133, 99, 151, 68]
模型原始回答（Agentic 最终 JSON，截取）： ```json
{
  "status": "answered",
  "claims": [
    {
      "statement": "本节推导实际上有一个前提",
      "evidence_ids": ["evi_5b7a349b408e"]
    },
    {
      "statement": "zi 已经中心化",
      "evidence_ids": ["evi_6ec845b0a2e3"]
    },
    {
      "statement": "即使xi 已进行
资料量： 4 → 8 个片段
模型调用次数：基线 1 次；Agentic Retrieval 3 次；合计 4 次。
结论：模型先规划并检查资料；若补查，则再次用 glm-4-flash 严格复核，最终答案遵守 evidence_id/quote 契约。

--- 代理式检索对照：核 PCA 的推导为什么要求映射后的样本先中心化？（模型=glm-4-flash）---


## 结果解读

审计输出保存每个案例的 plan、首轮 verify、repair（若发生）、repair 后 verify 和最终 answer contract 的原始响应与严格解析结果，同时保存实际检索顺序、证据合并结果和每阶段调用次数。答案阶段的模型 raw 只允许输出 `{status, claims}`：每条 claim 的 `statement` 必须由其 1..4 个 canonical quote 按顺序逐字拼接（只可移除空白）；parser hydrate 出 evidence 后确定性生成 `answer = ''.join(claim.statement)`，不再接受模型另写的 answer。资料不足时 raw 只能是 `status=insufficient, claims=[]`，parser 生成固定拒答“资料不足，无法可靠回答。”。Notebook 不把长 raw JSON 打印给读者；若任一阶段返回非法 JSON、字段不精确、covered/missing 规范化后重叠或引用未检索 evidence，执行会在该阶段阻断。


## Agentic RAG 的实现范围和边界

本页展示已经执行并保存 trace 的受约束 Agentic RAG：plan 是规划，retrieve 是工具执行和 observation，verify 是 reflection，repair 是一次预算明确的计划修正，repair 后会再次 verify。答案阶段 raw 只输出 `{status, claims}`；每条 claim 绑定 1..4 个 canonical quote，parser hydrate 后确定性拼接 `answer`，不足时生成固定拒答。最终 answer contract 负责 `{statement, evidence_ids, evidence}` 多证据绑定（`evidence_ids → [{evidence_id, page, quote}]`）。

它没有实现开放式 ReAct 的任意工具循环，也没有实现 Plan-then-Execute 的多轮动态重规划；这种限制让教程能逐步核对每个动作，并防止无限循环和不可解释的工具调用。因此本页支持的结论是“Agentic RAG 的检索编排闭环已经跑通”，不是“任意智能体任务都已解决”。本次真实运行保存的两个案例首轮资料均已足够，所以没有伪造 repair 或 insufficient；repair→reverify 分支仅由标记为 simulated/unit 的 mock/纯函数专项合约测试覆盖，不声称是真实模型实测。这里的 dense、BM25、hybrid 都针对同一份 `pumpkin_book.pdf`，结果不能外推成多资料来源路由。

模型 provenance 也要按事实阅读：baseline、plan、verify、repair（若触发）和 answer 都通过同一个 `call_glm_once` 使用 `glm-4-flash`；verify/repair 是同一模型的独立请求，不是跨模型语义复核。本页没有人工审核结论；C7 合成训练数据的记录明确 `human_verified=false`，因此模型生成或模型辅助检查都不能写成人工核验，也不能据此捏造 reviewer 的版本或 prompt。
